# Selective TSP/Orienteering problems

This combinatorial problems is aiming at finding the best tour visiting only a subset of nodes in a graph.
The nodes are organized as cluster and the vehicles should visit a node on each cluster. Therefore we're not in a case of a classical Traveling salesman problem.



# Solving imports

In [ ]:
import sys
import os
sys.path.append("../")

In [ ]:
from typing import Set, Hashable
import numpy as np
import scipy.spatial.distance as dist
from discrete_optimization.pickup_vrp.gpdp import GPDP, ProxyClass, build_pruned_problem
from discrete_optimization.pickup_vrp.solver.ortools_solver import ORToolsGPDP, local_search_metaheuristic_enum, first_solution_strategy_enum, \
    plot_ortools_solution, ParametersCost
import time
import matplotlib.pyplot as plt
import random
import scipy.spatial.distance as dist

# Creation of a handcrafted example

In [ ]:
def create_selective_tsp(nb_nodes=300, nb_vehicles=1, nb_clusters=30):
    number_vehicle = nb_vehicles
    nb_nodes_transportation = nb_nodes
    
    # non dummy nodes
    nodes_transportation = set(range(nb_nodes_transportation))
    
    # we stack the origin nodes after
    nodes_origin = {nb_nodes_transportation+i for i in range(number_vehicle)}
    # and destination nodes too 
    nodes_target = {nb_nodes_transportation+number_vehicle+i for i in range(number_vehicle)}
    list_pickup_deliverable = []
    origin_vehicle = {i: nb_nodes_transportation+i for i in range(number_vehicle)}
    target_vehicle = {i: nb_nodes_transportation+number_vehicle+i for i in range(number_vehicle)}
    # we don't include resources neither capacities
    resources_set = set()
    capacities = {i: {} for i in range(number_vehicle)}
    resources_flow_node = {i: {} for i in range(nb_nodes_transportation)}
    resources_flow_edges = {i: {j: {} for j in range(nb_nodes_transportation) if j != i}
                            for i in range(nb_nodes_transportation)}

    # real number of nodes in the problem definition 
    nb_nodes_real = nb_nodes_transportation+2*number_vehicle
    # we sample random 2D coordinates
    coordinates = np.random.randint(-20, 20, size=(nb_nodes_real,
                                                   2))
    distance_delta = dist.cdist(coordinates, coordinates)
    distance_delta = np.array(distance_delta, dtype=np.int32)
    distance_delta_dict = {i: {j: int(distance_delta[i, j])
                               for j in range(nb_nodes_real) if j != i}
                           for i in range(nb_nodes_real)}
    time_delta_dict = {i: {j: int(distance_delta_dict[i][j]/2)
                           for j in distance_delta_dict[i]}
                       for i in distance_delta_dict}
    from sklearn.cluster import KMeans
    nb_clusters = nb_clusters #int(nb_nodes_transportation/10)
    # compute clusters based on geographical positions.
    kmeans = KMeans(n_clusters=nb_clusters, random_state=0).fit(coordinates)
    labels = kmeans.labels_
    coordinates = {i: tuple(coordinates[i, :]) for i in range(coordinates.shape[0])}
    clusters_dict = {i: i+1 for i in range(nb_nodes_real)}
    print(clusters_dict)
    for i in clusters_dict:
        clusters_dict[i] = labels[i]  # random.randint(0, nb_clusters-1)
    for j in range(number_vehicle):
        clusters_dict[target_vehicle[j]] = max(labels)
        clusters_dict[origin_vehicle[j]] = min(labels)
    return GPDP(number_vehicle=number_vehicle,
                nodes_transportation=nodes_transportation,
                nodes_origin=nodes_origin,
                nodes_target=nodes_target,
                list_pickup_deliverable=list_pickup_deliverable,
                origin_vehicle=origin_vehicle,
                target_vehicle=target_vehicle,
                resources_set=resources_set,
                capacities=capacities,
                resources_flow_edges=resources_flow_edges,
                resources_flow_node=resources_flow_node,
                distance_delta=distance_delta_dict,
                time_delta=time_delta_dict,
                coordinates_2d=coordinates,
                clusters_dict=clusters_dict)


In [ ]:
gpdp = create_selective_tsp(nb_nodes=200, nb_vehicles=3, nb_clusters=40)

# Use tabou search

In [ ]:
ParametersCost??

In [ ]:
params_cost = [ParametersCost(dimension_name="Distance", 
                              global_span=True,
                              sum_over_vehicles=False, 
                              coefficient_vehicles=20)]

In [ ]:
solver = ORToolsGPDP(problem=gpdp,
                     factor_multiplier_distance=1,
                     factor_multiplier_time=1)
solver.init_model(one_visit_per_cluster=True,
                  one_visit_per_node=False,
                  include_time_dimension=True,
                  include_demand=True,
                  include_mandatory=True,
                  include_pickup_and_delivery=False,
                  parameters_cost=params_cost,
                  local_search_metaheuristic=local_search_metaheuristic_enum.GENERIC_TABU_SEARCH,
                  first_solution_strategy=first_solution_strategy_enum.UNSET,
                  time_limit=50,
                  n_solutions=10000)
results_tabu = solver.solve()
res_to_plot_tabu = min([r for r in results_tabu], key=lambda x: x[-1])
print("Best cost obtained = ", res_to_plot_tabu[-1])
plot_ortools_solution(res_to_plot_tabu, gpdp)
plt.show()

Use guided local search

In [ ]:
solver = ORToolsGPDP(problem=gpdp,
                     factor_multiplier_distance=1,
                     factor_multiplier_time=1)
solver.init_model(one_visit_per_cluster=True,
                  one_visit_per_node=False,
                  include_time_dimension=True,
                  include_demand=True,
                  include_mandatory=True,
                  include_pickup_and_delivery=False,
                  parameters_cost=params_cost,
                  local_search_metaheuristic=local_search_metaheuristic_enum.GUIDED_LOCAL_SEARCH,
                  first_solution_strategy=first_solution_strategy_enum.UNSET,
                  time_limit=50,
                  n_solutions=10000)
results_gls = solver.solve()
res_to_plot_gls = min([r for r in results_gls], key=lambda x: x[-1])
print("Best cost obtained = ", res_to_plot_gls[-1])
plot_ortools_solution(res_to_plot_gls, gpdp)
plt.show()

Use simulated annealing

In [ ]:
solver = ORToolsGPDP(problem=gpdp,
                     factor_multiplier_distance=1,
                     factor_multiplier_time=1)
solver.init_model(one_visit_per_cluster=True,
                  one_visit_per_node=False,
                  include_time_dimension=True,
                  include_demand=True,
                  include_mandatory=True,
                  include_pickup_and_delivery=False,
                  parameters_cost=params_cost,
                  local_search_metaheuristic=local_search_metaheuristic_enum.TABU_SEARCH,
                  first_solution_strategy=first_solution_strategy_enum.UNSET,
                  time_limit=50,
                  n_solutions=10000)
results_sa = solver.solve()
res_to_plot_sa = min([r for r in results_sa], key=lambda x: x[-1])
print("Best cost obtained = ", res_to_plot_sa[-1])
plot_ortools_solution(res_to_plot_sa, gpdp)
plt.show()

In [ ]:
print("Best cost obtained TABU = ", res_to_plot_tabu[-1])
print("Best cost obtained GLS = ", res_to_plot_gls[-1])
print("Best cost obtained SA = ", res_to_plot_sa[-1])
plot_ortools_solution(res_to_plot_tabu, gpdp)
plot_ortools_solution(res_to_plot_gls, gpdp)
plot_ortools_solution(res_to_plot_sa, gpdp)
plt.show()